# 09 — Revision Notes Generation

This notebook demonstrates the **RevisionNotesWorkflow** — a sequential pipeline that:
1. **Retrieves** relevant content chunks for a topic (hybrid retrieval)
2. **Generates** hierarchical revision notes via an LLM
3. **Validates** the output against Pydantic models

The output is structured as:
- **Topic** → Subtopics → Bullet points
- Key terms and definitions
- Formulae
- Mnemonics / memory aids

Each subtopic has an **importance level** (high / medium / low) to help prioritize study time.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.workflows.revision_notes import RevisionNotesWorkflow
from src.llm import LLMClient
from src.retrieval import Retriever
from models.output import RevisionNote, SubtopicNote

## Setup

Initialize the LLM client and Retriever. You'll need:
- A `.env` file with your API keys (GROQ_API_KEY, OPENROUTER_API_KEY, or GITHUB_TOKEN)
- A populated vector store and knowledge graph (run ingestion notebooks first)

In [ ]:
# Initialize components
# NOTE: Replace with your actual store initialization
from src.store import VectorStore, KnowledgeGraph

vector_store = VectorStore()
knowledge_graph = KnowledgeGraph()

retriever = Retriever(vector_store=vector_store, knowledge_graph=knowledge_graph)
llm_client = LLMClient()

print(f"Available LLM providers: {llm_client.available_providers}")

## Create the Workflow

In [ ]:
workflow = RevisionNotesWorkflow(
    retriever=retriever,
    llm_client=llm_client,
    top_k=8,  # Number of chunks to retrieve for context
)

print("RevisionNotesWorkflow initialized.")

## Generate Revision Notes

Provide a topic and the workflow will:
1. Retrieve relevant chunks from your knowledge base
2. Generate structured, hierarchical notes
3. Return a validated `RevisionNote` object

In [ ]:
topic = "Photosynthesis"
revision_note = workflow.generate(topic)

print(f"Topic: {revision_note.topic}")
print(f"Subtopics: {len(revision_note.subtopics)}")
print(f"Key terms: {len(revision_note.key_terms)}")
print(f"Formulae: {len(revision_note.formulae)}")
print(f"Mnemonics: {len(revision_note.mnemonics)}")

## Display Hierarchical Notes

Format the notes as readable, hierarchical bullets.

In [ ]:
def display_revision_notes(note: RevisionNote) -> None:
    """Pretty-print revision notes in hierarchical bullet format."""
    print(f"{'='*60}")
    print(f"📚 REVISION NOTES: {note.topic}")
    print(f"{'='*60}")
    
    # Subtopics with bullet points
    for subtopic in note.subtopics:
        importance_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
        emoji = importance_emoji.get(subtopic.importance, "⚪")
        print(f"\n{emoji} {subtopic.title} [{subtopic.importance.upper()}]")
        print(f"{'─'*40}")
        for point in subtopic.points:
            print(f"  • {point}")
    
    # Key terms
    if note.key_terms:
        print(f"\n{'─'*60}")
        print("📖 KEY TERMS:")
        for term in note.key_terms:
            print(f"  ▸ {term}")
    
    # Formulae
    if note.formulae:
        print(f"\n{'─'*60}")
        print("🔢 FORMULAE:")
        for formula in note.formulae:
            print(f"  ∴ {formula}")
    
    # Mnemonics
    if note.mnemonics:
        print(f"\n{'─'*60}")
        print("🧠 MNEMONICS:")
        for mnemonic in note.mnemonics:
            print(f"  💡 {mnemonic}")
    
    print(f"\n{'='*60}")


display_revision_notes(revision_note)

## Export to JSON

The `RevisionNote` model supports serialization for storage or further processing.

In [ ]:
import json

# Export as JSON
json_output = revision_note.to_json()
print(json_output[:500], "...")

## Filter by Importance

You can filter subtopics by importance level for focused revision sessions.

In [ ]:
# Get only high-importance subtopics
high_priority = [s for s in revision_note.subtopics if s.importance == "high"]

print(f"High-priority subtopics ({len(high_priority)}):")
for subtopic in high_priority:
    print(f"  🔴 {subtopic.title}")
    for point in subtopic.points[:2]:  # Show first 2 points
        print(f"     • {point}")

## Summary

The `RevisionNotesWorkflow` provides:
- **Hierarchical organization**: topic → subtopics → bullet points
- **Importance levels**: high/medium/low for study prioritization
- **Key terms**: definitions for quick reference
- **Formulae**: relevant equations in plain text
- **Mnemonics**: memory aids for retention
- **Pydantic validation**: guaranteed output structure
- **Fallback retrieval**: hybrid → semantic if needed